In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('./sample30.csv')
print(df.head())
df['user_id_encoded'] = df['reviews_username'].astype('category').cat.codes
df['item_id_encoded'] = df['id'].astype('category').cat.codes
df['relation_encoded'] = 0

df['rating'] = df['reviews_rating']
df['reviews_text'] = df['reviews_text'].fillna("")




                     id            brand  \
0  AV13O1A8GV-KLJ3akUyj  Universal Music   
1  AV14LG0R-jtxr-f38QfS         Lundberg   
2  AV14LG0R-jtxr-f38QfS         Lundberg   
3  AV16khLE-jtxr-f38VFn              K-Y   
4  AV16khLE-jtxr-f38VFn              K-Y   

                                          categories  \
0  Movies, Music & Books,Music,R&b,Movies & TV,Mo...   
1  Food,Packaged Foods,Snacks,Crackers,Snacks, Co...   
2  Food,Packaged Foods,Snacks,Crackers,Snacks, Co...   
3  Personal Care,Medicine Cabinet,Lubricant/Sperm...   
4  Personal Care,Medicine Cabinet,Lubricant/Sperm...   

                         manufacturer  \
0  Universal Music Group / Cash Money   
1                            Lundberg   
2                            Lundberg   
3                                 K-Y   
4                                 K-Y   

                                         name              reviews_date  \
0   Pink Friday: Roman Reloaded Re-Up (w/dvd)  2012-11-30T06:21:45.000Z   
1

In [5]:
df['user_id_encoded'].describe()

count    30000.000000
mean     12451.277767
std       7169.785924
min         -1.000000
25%       6245.750000
50%      12459.500000
75%      18624.250000
max      24913.000000
Name: user_id_encoded, dtype: float64

In [6]:
len(df)

30000

In [4]:
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

I0000 00:00:1790069784.390259   39213 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790069784.430501   39213 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenizati

In [5]:
def build_joint_explainability_model(num_entities, num_relations, vocab_size, max_seq_len = 128, embed_dim = 64):
    input_ids = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name = 'input_ids')
    attn_mask = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name = 'attention_mask')


    bert = TFBertModel.from_pretrained('bert-base-uncased')
    bert.trainable = False
    bert_output = bert(input_ids = input_ids, attention_mask = attn_mask)
    bert_polled = bert_output.pooler_output

    head_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'head_idx')
    # rel_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'rel_idx')
    tail_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'tail_idx')
    brand_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'brand_idx')
    cat_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'cat_idx')


    entity_emb_layer = tf.keras.layers.Embedding(input_dim = num_entities, output_dim = embed_dim, name = 'entity_emb')
    # relation_emb_layer = tf.keras.layers.Embedding(input_dim = num_relations, output_dim = embed_dim, name = 'relation_emb')



    h_emb = tf.keras.layers.Flatten()(entity_emb_layer(head_idx))
    # r_emb = tf.keras.layers.Flatten()(relation_emb_layer(rel_idx))
    t_emb = tf.keras.layers.Flatten()(entity_emb_layer(tail_idx))
    b_emb = tf.keras.layers.Flatten()(entity_emb_layer(brand_idx))
    c_emb = tf.keras.layers.Flatten()(entity_emb_layer(cat_idx))

    merged = tf.keras.layers.concatenate([h_emb, t_emb, b_emb, c_emb])


    fusion_dense = tf.keras.layers.Dense(256, activation= 'relu', name = 'fusion_dense')(merged)
    fusion_state = tf.keras.layers.Dropout(0.2)(fusion_dense)


    rating_output = tf.keras.layers.Dense(1, activation= 'linear', name = 'rating_output')(fusion_state)




    decoder_inputs = tf.keras.layers.Input(shape = (None,), dtype="int32", name = 'decoder_inputs')

    lstm_initial_state = tf.keras.layers.Dense(512, activation= 'tanh', name = 'lstm_initial_state')(fusion_state)

    decoder_embedding = tf.keras.layers.Embedding(input_dim = vocab_size, output_dim = 256, name = 'decoder_embedding')(decoder_inputs)


    decoder_lstm = tf.keras.layers.LSTM(512,return_sequences=True, activation= 'tanh', name = 'justification_lstm')
    lstm_out = decoder_lstm(decoder_embedding, initial_state=[lstm_initial_state, lstm_initial_state])


    justification_output = tf.keras.layers.Dense(vocab_size, name = 'justification_output' )(lstm_out)

    model = tf.keras.Model(
        inputs = [head_idx,          # User ID
            tail_idx,          # Item ID
            brand_idx,         # Brand ID
            cat_idx,           # Category ID
            input_ids,         # BERT token IDs (Encoder)
            attn_mask,    # BERT padding mask (Encoder)
            decoder_inputs     # LSTM shifted targets (Decoder)
        ],
        outputs = [rating_output, justification_output],
        name="Explainable_Hybrid_Recommender",
    )

    return model

In [7]:
brands_unique = df['brand'].dropna().unique()
max_user_id = df['user_id_encoded'].max()
max_item_id = df['item_id_encoded'].max()

print('df[useridencoded] ', df['user_id_encoded'])
print('maxuserid ', max_user_id)
print('maxitemid ', max_item_id)
num_entities = max(max_user_id, max_item_id)

brands_map = {brand : idx + num_entities + 1 for idx, brand in enumerate(brands_unique)}

df[useridencoded]  0        11440
1         6974
2         6974
3        19327
4        24205
         ...  
29995    13350
29996    20518
29997    13893
29998    13062
29999    12262
Name: user_id_encoded, Length: 30000, dtype: int16
maxuserid  24913
maxitemid  270


In [10]:
pd.DataFrame.from_dict(brands_map)

ValueError: If using all scalar values, you must pass an index

In [6]:
vocab_size = tokenizer.vocab_size
max_user_id = df['user_id_encoded'].max()
max_item_id = df['item_id_encoded'].max()

print('df[useridencoded] ', df['user_id_encoded'])
print('maxuserid ', max_user_id)
print('maxitemid ', max_item_id)
num_entities = max(max_user_id, max_item_id)

brands_unique = df['brand'].dropna().unique()

brands_map = {brand : idx + num_entities + 1 for idx, brand in enumerate(brands_unique)}
df['brand_encoded'] = df['brand'].map(brands_map)
print(df['brand_encoded'])
num_entities = max(num_entities, df['brand_encoded'].max())


cats_unique = df['categories'].dropna().unique()

cats_map = {category : idx + num_entities for idx, category in enumerate(cats_unique)}

df['category_encoded'] = df['categories'].map(cats_map)


num_entities = max(num_entities, df['category_encoded'].max()) + 1
num_relations = 3


model = build_joint_explainability_model(num_entities, num_relations, vocab_size, 128, 64)


model.summary()


model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate= 2e-5, clipnorm = 1.0),
              loss = {
                  'rating_output' : 'mse',
                  'justification_output': tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              },
              loss_weights = {
                  'rating_output' : 1,
                  'justification_output' : 0.5
              },
              metrics = {'rating_output': 'mae'},
              )



df[useridencoded]  0        11440
1         6974
2         6974
3        19327
4        24205
         ...  
29995    13350
29996    20518
29997    13893
29998    13062
29999    12262
Name: user_id_encoded, Length: 30000, dtype: int16
maxuserid  24913
maxitemid  270
0        24914
1        24915
2        24915
3        24916
4        24916
         ...  
29995    25127
29996    25127
29997    25127
29998    25127
29999    25127
Name: brand_encoded, Length: 30000, dtype: int16


I0000 00:00:1790069792.135241   39213 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5262 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification mode

Model: "Explainable_Hybrid_Recommender"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 head_idx (InputLayer)       [(None, 1)]                  0         []                            
                                                                                                  
 tail_idx (InputLayer)       [(None, 1)]                  0         []                            
                                                                                                  
 brand_idx (InputLayer)      [(None, 1)]                  0         []                            
                                                                                                  
 cat_idx (InputLayer)        [(None, 1)]                  0         []                            
                                                                     

In [7]:
import numpy as np
from transformers import BertTokenizer


def create_tf_dataset(df, tokenizer, max_len = 128, batch_size = 16, is_training = True):
    print(f'Tokenizinf {len(df)}  Reviews')
    tokenized = tokenizer(
        df['reviews_text'].tolist(),
        truncation = True,
        padding = 'max_length',
        max_length = max_len,
        return_tensors = 'np',
    )




    input_ids = tokenized['input_ids']
    attn_mask = tokenized['attention_mask']


    decoder_inputs = input_ids[:, :-1]
    justification_targets = input_ids[:, 1:]

    inputs = {
        'input_ids': input_ids,
        'attention_mask': attn_mask,
        'decoder_inputs': decoder_inputs,
        "head_idx": df['user_id_encoded'].values.reshape(-1, 1),
        "rel_idx": df['relation_encoded'].values.reshape(-1, 1),
        "tail_idx": df['item_id_encoded'].values.reshape(-1, 1),
        "brand_idx": df['brand_encoded'].values.reshape(-1, 1),
        "cat_idx": df['category_encoded'].values.reshape(-1, 1),

    }

    targets = {
        'rating_output':df['rating'].values.reshape(-1, 1),
        "justification_output": justification_targets
    }
    justification_weights = (justification_targets != 0).astype(np.float32)
    rating_weight = np.ones((len(df), 1), dtype = np.float32)
    sample_weights = {
        'rating_output': rating_weight,
        'justification_output': justification_weights
    }
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets, sample_weights))
    if is_training:
        dataset = dataset.shuffle(buffer_size = len(df), seed = 42)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
df_shuffled = df.sample(frac = 1, random_state = 42).reset_index(drop = True)
split_idx = int(len(df_shuffled) * 0.8)


train_idx = df_shuffled.iloc[:split_idx]
val_idx = df_shuffled.iloc[split_idx:]

train_dataset = create_tf_dataset(train_idx, tokenizer, max_len = 128, batch_size = 16, is_training = True)

val_dataset = create_tf_dataset(val_idx, tokenizer, max_len = 128, batch_size = 16, is_training = False)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights=True)],
)








/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Tokenizinf 24000  Reviews
Tokenizinf 6000  Reviews
Epoch 1/15


/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/tf_keras/src/engine/functional.py:661: UserWarning: Input dict contained keys ['rel_idx'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)
I0000 00:00:1790069805.828973   39297 cuda_dnn.cc:461] Loaded cuDNN version 92501
I0000 00:00:1790069806.349152   39289 service.cc:153] XLA service 0x7b98be15f150 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790069806.349178   39289 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5050 Laptop GPU, Compute Capability 12.0a (Driver: 13.4.0; Runtime: 13.0.0; Toolkit: 13.0.0; DNN: 9.25.1)
I0000 00:00:1790069806.355141   39289 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790069806.434971   39289 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most 

1500/1500 [==============================] - 110s 71ms/step - loss: 15.6704 - rating_output_loss: 14.5917 - justification_output_loss: 2.1574 - rating_output_mae: 3.6144 - val_loss: 7.2653 - val_rating_output_loss: 6.2885 - val_justification_output_loss: 1.9536 - val_rating_output_mae: 2.1735
Epoch 2/15
1500/1500 [==============================] - 89s 60ms/step - loss: 3.8935 - rating_output_loss: 2.9338 - justification_output_loss: 1.9195 - rating_output_mae: 1.2816 - val_loss: 2.3822 - val_rating_output_loss: 1.4084 - val_justification_output_loss: 1.9476 - val_rating_output_mae: 0.7849
Epoch 3/15
1500/1500 [==============================] - 90s 60ms/step - loss: 2.1116 - rating_output_loss: 1.1537 - justification_output_loss: 1.9157 - rating_output_mae: 0.7245 - val_loss: 1.8956 - val_rating_output_loss: 0.9227 - val_justification_output_loss: 1.9457 - val_rating_output_mae: 0.6341
Epoch 4/15
1500/1500 [==============================] - 89s 60ms/step - loss: 1.8279 - rating_output_l

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction



def evaluate_nlg_performance(model, val_dataset, tokenizer):
    print('Evaluating NLG')

    bleu_scores = []
    smoother = SmoothingFunction().method1
    for batch_inputs, batch_targets, batch_weights in val_dataset:
        predictions = model(batch_inputs, training = False)
        justification_logits = predictions[1]
        pred_token_ids = tf.argmax(justification_logits, axis = -1).numpy()
        true_token_ids = batch_targets['justification_output'].numpy()

        for i in range(0, len(true_token_ids)):
            true_text = tokenizer.decode(true_token_ids[i], skip_special_tokens=True).split()
            pred_text = tokenizer.decode(pred_token_ids[i], skip_special_tokens=True).split()
            if len(true_text) > 0:
               score = sentence_bleu([true_text], pred_text, smoothing_function=smoother)
               bleu_scores.append(score)
    final_score = np.mean(bleu_scores)
    print('Final Score = ', final_score)
    print("\n--- Example Generation ---")
    print(f"Target: {' '.join(true_text)}")
    print(f"Output: {' '.join(pred_text)}")

    return final_score
val_bleu = evaluate_nlg_performance(model, val_dataset, tokenizer)
print(f'BLEU Score = {val_bleu}')




In [ ]:
model.save_weights('baseline_hybrid_weights.h5')

In [ ]:
def generate_justification(model,user_id, item_id,  tokenizer, max_length = 30):
    generated_sequence = [tokenizer.cls_token_id]
    head_tensor = tf.constant([[user_id]])
    tail_tensor = tf.constant([[item_id]])


    for _ in range(max_length):
        decoder_input = tf.constant([generated_sequence])
        dummy_input_ids = tf.zeros((1, 128), dtype = tf.int32)
        predictions = model(
            [dummy_input_ids, dummy_input_ids, head_tensor, rel_tensor,  tail_tensor, decoder_input],
            training = False
        )

        next_token_logits = predictions[1][0, -1, :]
        next_token_id = tf.argmax(next_token_logits, axis = -1).numpy()

        if next_token_id == tokenizer.sep_token_id:
            break
        generated_sequence.append(next_token_id)
        return tokenizer.decode(generated_sequence, skip_special_tokens=True)